# Customer Churn Intelligence — Exploratory Data Analysis (EDA)

## 1. Executive Summary & Problem Context
Customer churn poses a substantial financial risk to subscription-based telecommunications providers. Customer acquisition costs typically exceed retention costs by a factor of 5x to 7x. Identifying the primary friction points and signals correlated with subscriber defection allows customer success and marketing teams to deploy proactive, high-value retention initiatives.

### Dataset Overview
- **Source:** IBM Telco Customer Churn (blastchar/telco-customer-churn)
- **Target Variable:** `Churn` (Binary: Yes / No)
- **Key Feature Categories:** Demographic attributes, subscribed telephone & internet services, account contracts, billing methods, and financial charges.

> **Methodological Notice:** All patterns observed in this analysis represent empirical statistical associations in historical data. **Correlation does not imply causation**. Interventions should be validated through randomized controlled trials (A/B testing) rather than assuming single-feature manipulation will eliminate churn.

In [1]:
import sys
from pathlib import Path

# Ensure project root is accessible
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data
from src.eda import set_plot_aesthetics

set_plot_aesthetics()
print("Libraries loaded successfully!")

## 2. Data Ingestion & Quality Validation
We load the authentic dataset via `src.data_loader`, which performs rigorous schema integrity checks, detects duplicate customer records, safely parses `TotalCharges` from text to numeric, and investigates records with zero tenure.

In [2]:
df, quality_report = load_raw_data(ROOT_DIR / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f"Dataset Shape: {df.shape[0]:,} rows by {df.shape[1]} columns")
print(f"Duplicate Customer IDs: {quality_report['duplicate_customer_ids']}")
print(f"Blank TotalCharges Count: {quality_report['blank_total_charges_count']}")
print(f"Zero-Tenure Blank Match: {quality_report['all_blank_total_charges_have_zero_tenure']}")
df.head(5)

## 3. Churn Distribution & Class Imbalance
Let us examine the balance of our target variable `Churn`.

In [3]:
counts = df["Churn"].value_counts()
percentages = df["Churn"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(["Retained (No)", "Churned (Yes)"], counts.values, color=["#2563EB", "#E11D48"], width=0.5)
for b, pct, cnt in zip(bars, percentages.values, counts.values):
    ax.annotate(f"{cnt:,}\n({pct:.1f}%)", xy=(b.get_x() + b.get_width() / 2, b.get_height()),
                xytext=(0, 5), textcoords="offset points", ha="center", fontweight="bold")
ax.set_title("Customer Churn Distribution (Class Imbalance)")
ax.set_ylabel("Number of Customers")
ax.set_ylim(0, max(counts.values) * 1.15)
plt.show()

### Observation & Business Implication:
- 26.5% of customers have churned (1,869 subscribers), while 73.5% (5,174 subscribers) remained.
- This class imbalance means **Accuracy is an inadequate metric**: a naive dummy baseline predicting 'No' for all users achieves 73.5% accuracy but detects 0% of churners.
- Models must be assessed using **ROC-AUC, Precision-Recall AUC, Recall, and F1-Score**.

## 4. Churn by Contract Commitment
How does the contractual commitment length relate to subscriber departure?

In [4]:
ct_contract = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
ct_contract = ct_contract.reindex(["Month-to-month", "One year", "Two year"])

fig, ax = plt.subplots(figsize=(7, 4.5))
ct_contract.plot(kind="bar", stacked=False, color=["#2563EB", "#E11D48"], ax=ax, width=0.6)
ax.set_title("Churn Rate Across Contract Categories")
ax.set_ylabel("Percentage within Contract Category (%)")
ax.set_ylim(0, 100)
plt.xticks(rotation=0)
plt.legend(["Retained (No)", "Churned (Yes)"])
plt.show()

for c in ct_contract.index:
    print(f"{c}: Churn Rate = {ct_contract.loc[c, 'Yes']:.1f}%")

### Observation:
- Month-to-month subscribers experience an alarming **42.7% churn rate**.
- One-year contracts drop churn to **11.3%**, and two-year commitments reduce it to just **2.8%**.
- **Business Takeaway:** Transitioning month-to-month subscribers to annual agreements (even at an introductory promotional rate) represents one of the highest-leverage retention tactics.

## 5. Tenure Dynamics: When Do Customers Depart?

In [5]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, multiple="dodge", palette=["#2563EB", "#E11D48"], shrink=0.85, ax=ax)
ax.set_title("Customer Tenure Distribution by Churn Status")
ax.set_xlabel("Tenure with Provider (Months)")
ax.set_ylabel("Number of Customers")
plt.legend(["Churned (Yes)", "Retained (No)"])
plt.show()

### Observation:
- Churn is heavily front-loaded: highest hazard rate is in the **first 1 to 5 months**.
- Customers who surpass the 24-month mark display substantial loyalty and retention stability.
- **Business Takeaway:** Onboarding experience and early lifecycle check-ins (days 30, 60, 90) are paramount to building habituation.

## 6. Internet Service Technology & Financial Charges

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Churn by Internet Service
ct_internet = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
ct_internet.plot(kind="bar", color=["#2563EB", "#E11D48"], ax=ax1, width=0.6)
ax1.set_title("Churn by Internet Service Technology")
ax1.set_ylabel("Percentage (%)")
ax1.set_ylim(0, 100)
ax1.set_xticklabels(ct_internet.index, rotation=0)
ax1.legend(["Retained (No)", "Churned (Yes)"])

# Monthly Charges Density
sns.kdeplot(data=df[df["Churn"] == "No"], x="MonthlyCharges", label="Retained (No)", fill=True, color="#2563EB", alpha=0.3, ax=ax2)
sns.kdeplot(data=df[df["Churn"] == "Yes"], x="MonthlyCharges", label="Churned (Yes)", fill=True, color="#E11D48", alpha=0.3, ax=ax2)
ax2.set_title("Monthly Charges ($) Distribution")
ax2.set_xlabel("Monthly Charges (USD)")
ax2.legend()

plt.tight_layout()
plt.show()

### Observation:
- **Fiber optic subscribers experience a 41.9% churn rate**, compared to 19.0% for DSL and 7.4% for no internet.
- Fiber optic plans carry higher monthly rates ($70-$105). Customers paying over $75/month have visibly higher probability of churn.
- Price sensitivity combined with service reliability complaints may drive this vulnerability.

## 7. Numerical Feature Correlation Matrix

In [7]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
corr_df = df[num_cols].copy()
corr_df["Churn_Binary"] = (df["Churn"] == "Yes").astype(int)

corr = corr_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="Blues", center=0, ax=ax, square=True)
ax.set_title("Pearson Correlation Matrix")
plt.show()

## 8. Summary of Findings & Limitations

### Key Empirical Findings:
1. **Contract Structure:** Month-to-month contracts are the strongest single predictor of churn risk.
2. **Tenure Curve:** Customer attrition risk decreases exponentially with relationship duration.
3. **Technology & Pricing:** High monthly charges ($75+) and Fiber Optic service correlate with higher defection.
4. **Payment Channel:** Electronic checks show substantially higher churn rates than automated debit/credit billing.

### Methodological Limitations:
- **Cross-Sectional Snapshot:** The dataset captures a single static snapshot rather than time-series event logs (e.g., ticket submissions, service outages, speed degradation).
- **Absence of Customer Sentiment Data:** Net Promoter Score (NPS), CSAT ratings, or competitor pricing in the subscriber's geographical zip code are not captured.
- **Non-Causality:** High feature correlations must not be mistaken for verified levers without controlled intervention testing.